In [1]:
!pip install xgboost

In [2]:
# Manipulación de datos
import pandas as pd
import numpy as np

# Dataset
from sklearn.datasets import load_wine

# División de datos
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV

# Modelos
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier

# Métricas
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# KS
from scipy.stats import ks_2samp

# Gráficos
import matplotlib.pyplot as plt
import seaborn as sns

print("Librerías cargadas correctamente")

Librerías cargadas correctamente


In [3]:
wine = load_wine()

print("Cantidad de registros:", wine.data.shape[0])
print("Cantidad de variables:", wine.data.shape[1])
print("Clases:", wine.target_names)

Cantidad de registros: 178
Cantidad de variables: 13
Clases: ['class_0' 'class_1' 'class_2']


In [4]:
wine = load_wine()

X = pd.DataFrame(
    wine.data,
    columns=wine.feature_names
)

y = pd.Series(wine.target)

print(X.head())
print(y.head())

   alcohol  malic_acid   ash  alcalinity_of_ash  magnesium  total_phenols  \
0    14.23        1.71  2.43               15.6      127.0           2.80   
1    13.20        1.78  2.14               11.2      100.0           2.65   
2    13.16        2.36  2.67               18.6      101.0           2.80   
3    14.37        1.95  2.50               16.8      113.0           3.85   
4    13.24        2.59  2.87               21.0      118.0           2.80   

   flavanoids  nonflavanoid_phenols  proanthocyanins  color_intensity   hue  \
0        3.06                  0.28             2.29             5.64  1.04   
1        2.76                  0.26             1.28             4.38  1.05   
2        3.24                  0.30             2.81             5.68  1.03   
3        3.49                  0.24             2.18             7.80  0.86   
4        2.69                  0.39             1.82             4.32  1.04   

   od280/od315_of_diluted_wines  proline  
0                  

In [5]:
# Convertir el problema a clasificación binaria
# Clase 0 queda como 0
# Clases 1 y 2 se agrupan como 1

y_bin = np.where(y == 0, 0, 1)

print("Clase 0:", sum(y_bin == 0))
print("Clase 1:", sum(y_bin == 1))

Clase 0: 59
Clase 1: 119


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_bin,
    test_size=0.30,
    random_state=42,
    stratify=y_bin
)

print("Entrenamiento:", X_train.shape)
print("Validación:", X_test.shape)

Entrenamiento: (124, 13)
Validación: (54, 13)


In [7]:
ridge = LogisticRegression(
    penalty='l2',
    solver='liblinear'
)

ridge.fit(X_train, y_train)

pred_ridge = ridge.predict(X_test)
prob_ridge = ridge.predict_proba(X_test)[:,1]

print("Modelo Ridge entrenado correctamente")

Modelo Ridge entrenado correctamente


In [8]:
print("Accuracy:", accuracy_score(y_test, pred_ridge))
print("Precision:", precision_score(y_test, pred_ridge))
print("Recall:", recall_score(y_test, pred_ridge))
print("F1:", f1_score(y_test, pred_ridge))
print("AUC:", roc_auc_score(y_test, prob_ridge))

Accuracy: 0.9814814814814815
Precision: 1.0
Recall: 0.9722222222222222
F1: 0.9859154929577465
AUC: 1.0


In [9]:
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf.fit(X_train, y_train)

pred_rf = rf.predict(X_test)
prob_rf = rf.predict_proba(X_test)[:,1]

print("Accuracy:", accuracy_score(y_test, pred_rf))
print("Precision:", precision_score(y_test, pred_rf))
print("Recall:", recall_score(y_test, pred_rf))
print("F1:", f1_score(y_test, pred_rf))
print("AUC:", roc_auc_score(y_test, prob_rf))

Accuracy: 0.9629629629629629
Precision: 0.9722222222222222
Recall: 0.9722222222222222
F1: 0.9722222222222222
AUC: 0.9984567901234568


In [10]:
xgb = XGBClassifier(
    eval_metric='logloss',
    random_state=42
)

xgb.fit(X_train, y_train)

pred_xgb = xgb.predict(X_test)
prob_xgb = xgb.predict_proba(X_test)[:,1]

print("Accuracy:", accuracy_score(y_test, pred_xgb))
print("Precision:", precision_score(y_test, pred_xgb))
print("Recall:", recall_score(y_test, pred_xgb))
print("F1:", f1_score(y_test, pred_xgb))
print("AUC:", roc_auc_score(y_test, prob_xgb))

Accuracy: 0.9629629629629629
Precision: 0.9722222222222222
Recall: 0.9722222222222222
F1: 0.9722222222222222
AUC: 0.9876543209876543


In [11]:
mlp = MLPClassifier(
    hidden_layer_sizes=(20,),
    max_iter=1000,
    random_state=42
)

mlp.fit(X_train, y_train)

pred_mlp = mlp.predict(X_test)
prob_mlp = mlp.predict_proba(X_test)[:,1]

print("Accuracy:", accuracy_score(y_test, pred_mlp))
print("Precision:", precision_score(y_test, pred_mlp))
print("Recall:", recall_score(y_test, pred_mlp))
print("F1:", f1_score(y_test, pred_mlp))
print("AUC:", roc_auc_score(y_test, prob_mlp))

Accuracy: 0.9074074074074074
Precision: 0.8780487804878049
Recall: 1.0
F1: 0.935064935064935
AUC: 0.9182098765432098


In [13]:
rf_acc = accuracy_score(y_test, pred_rf)
rf_prec = precision_score(y_test, pred_rf)
rf_rec = recall_score(y_test, pred_rf)
rf_f1 = f1_score(y_test, pred_rf)
rf_auc = roc_auc_score(y_test, prob_rf)

print(rf_acc, rf_prec, rf_rec, rf_f1, rf_auc)

0.9629629629629629 0.9722222222222222 0.9722222222222222 0.9722222222222222 0.9984567901234568


In [14]:
xgb_acc = accuracy_score(y_test, pred_xgb)
xgb_prec = precision_score(y_test, pred_xgb)
xgb_rec = recall_score(y_test, pred_xgb)
xgb_f1 = f1_score(y_test, pred_xgb)
xgb_auc = roc_auc_score(y_test, prob_xgb)

print(xgb_acc, xgb_prec, xgb_rec, xgb_f1, xgb_auc)

0.9629629629629629 0.9722222222222222 0.9722222222222222 0.9722222222222222 0.9876543209876543


In [15]:
mlp_acc = accuracy_score(y_test, pred_mlp)
mlp_prec = precision_score(y_test, pred_mlp)
mlp_rec = recall_score(y_test, pred_mlp)
mlp_f1 = f1_score(y_test, pred_mlp)
mlp_auc = roc_auc_score(y_test, prob_mlp)

print(mlp_acc, mlp_prec, mlp_rec, mlp_f1, mlp_auc)

0.9074074074074074 0.8780487804878049 1.0 0.935064935064935 0.9182098765432098


In [16]:
tabla = pd.DataFrame({
    'Modelo':['Ridge','Random Forest','XGBoost','Red Neuronal'],
    'Accuracy':[0.9815, rf_acc, xgb_acc, mlp_acc],
    'Precision':[1.0, rf_prec, xgb_prec, mlp_prec],
    'Recall':[0.9722, rf_rec, xgb_rec, mlp_rec],
    'F1':[0.9859, rf_f1, xgb_f1, mlp_f1],
    'AUC':[1.0, rf_auc, xgb_auc, mlp_auc]
})

tabla

,Modelo,Accuracy,Precision,Recall,F1,AUC
0,Ridge,0.981500,1.000000,0.972200,0.985900,1.000000
1,Random Forest,0.962963,0.972222,0.972222,0.972222,0.998457
2,XGBoost,0.962963,0.972222,0.972222,0.972222,0.987654
3,Red Neuronal,0.907407,0.878049,1.000000,0.935065,0.918210


In [31]:
!git init
!git config --global user.name "Niro2595"
!git config --global user.email "nicolas.rojasroj@mayor.cl"

Reinitialized existing Git repository in /content/.git/


In [35]:
%%writefile README.md
# Proyecto Wine

Clasificación binaria usando:
- Ridge
- Random Forest
- XGBoost
- Red Neuronal

Writing README.md


In [36]:
!ls

README.md  sample_data


In [37]:
!git add README.md
!git commit -m "Primer commit"

[master (root-commit) 6ad1e1b] Primer commit
 1 file changed, 7 insertions(+)
 create mode 100644 README.md


In [38]:
!ls -la

total 24
drwxr-xr-x 1 root root 4096 Jun 15 19:18 .
drwxr-xr-x 1 root root 4096 Jun 15 18:35 ..
drwxr-xr-x 4 root root 4096 Jun  4 13:32 .config
drwxr-xr-x 8 root root 4096 Jun 15 19:19 .git
-rw-r--r-- 1 root root   97 Jun 15 19:18 README.md
drwxr-xr-x 1 root root 4096 Jun  4 13:32 sample_data


In [40]:
!git add README.md

In [42]:
!git status

On branch master
Untracked files:
  (use "git add <file>..." to include in what will be committed)
	.config/
	sample_data/

nothing added to commit but untracked files present (use "git add" to track)
